Search for Instru test in raw text

In [1]:
# -*- coding: utf-8 -*-
"""
Scan non-build Gradle(.kts) files for Android instrumentation *build* signals
ONLY for repos whose Build_pred == 0 (from stratified_sample_moe10.csv).

Inputs:
- CSV:
    C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\stratified_sample_moe10.csv
- Configs:
    C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files

Outputs (saved to the Stratified Sample folder):
- instru_signals_in_non_build_gradle_files_Build_pred0.csv
- _diagnostics_unmatched_nonbuild_gradle_files.csv  (files we saw but couldn’t map to Build_pred==0 repos)
"""

from pathlib import Path
import re
import pandas as pd

# ---------- Paths ----------
SAMPLE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample")
SAMPLE_FILE = SAMPLE_DIR / "stratified_sample_moe_using RandomSample65.csv"
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")

OUT_CSV = SAMPLE_DIR / "instru_signals_in_non_build_gradle_files_Build_pred0.csv"
OUT_DIAG = SAMPLE_DIR / "_diagnostics_unmatched_nonbuild_gradle_files.csv"

# ---------- Instrumentation *build* signals (scan raw text incl. comments) ----------
SIGNAL_PATTERNS = {
    # Core wiring
    "testInstrumentationRunner": r'\btestInstrumentationRunner\s*["\']',
    "androidTest deps (explicit)": r'\bandroidTest(?:Implementation|Compile)\s*["\']',
    # Common libraries that imply instrumentation
    "androidx.test family": r'\bandroidx\.test\b',
    "legacy support test": r'\bcom\.android\.support\.test\b',
    "espresso": r'\bespresso(?:-core)?\b',
    "uiautomator": r'\buiautomator\b',
    # Gradle DSL hints frequently used with device tests
    "android { testOptions ... }": r'android\s*\{[^}]*\btestOptions\b',
    "managedDevices/deviceGroups": r'\b(managedDevices|deviceGroups)\b',
    "sourceSets { androidTest ... }": r'sourceSets\s*\{[^}]*\bandroidTest\b',
    "kaptAndroidTest": r'\bkaptAndroidTest\b',
    # Optional text cues
    "connectedAndroidTest (text cue)": r'\bconnectedAndroidTest\b',
    "connectedCheck (text cue)": r'\bconnectedCheck\b',
}

# ---------- Helpers ----------
def load_sample_csv(sample_file: Path) -> pd.DataFrame:
    if not sample_file.exists():
        raise FileNotFoundError(f"Input CSV not found: {sample_file}")
    df = pd.read_csv(sample_file)
    df.columns = [str(c).strip() for c in df.columns]
    return df

def get_build_pred_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if c.lower() == "build_pred":
            return c
    raise ValueError("Column 'Build_pred' not found in the CSV.")

def normalize_full_name_value(val: str) -> str:
    """
    Normalize a CSV repo field to 'owner/repo' (lowercase).
    Accepts 'owner/repo', 'owner.repo', or a GitHub URL.
    """
    if not val:
        return ""
    s = str(val).strip()
    s = s.replace("\\", "/")
    s_low = s.lower()

    # If it's a GitHub URL
    if "github.com" in s_low:
        # grab the last two path segments
        parts = [p for p in s_low.split("/") if p]
        if len(parts) >= 2:
            owner, repo = parts[-2], parts[-1]
            # strip .git if present
            repo = repo[:-4] if repo.endswith(".git") else repo
            return f"{owner}/{repo}"

    # If it looks like owner/repo already
    if "/" in s:
        parts = s.split("/")
        if len(parts) >= 2:
            return f"{parts[-2].lower()}/{parts[-1].lower()}"

    # If it looks like owner.repo
    if "." in s and "__" not in s:
        parts = s.split(".")
        if len(parts) >= 2:
            return f"{parts[-2].lower()}/{parts[-1].lower()}"

    # fallback: return lower
    return s_low

def normalize_full_name_column(df: pd.DataFrame) -> pd.Series:
    """
    Return a Series of normalized 'owner/repo' values from any of these columns:
    full_name, repo, repository, owner_repo, name, file_name/filename/file (owner.repo__ prefix).
    """
    cmap = {c.lower(): c for c in df.columns}
    # Try direct repo name columns first
    for key in ("full_name", "repo", "repository", "owner_repo", "name"):
        if key in cmap:
            return df[cmap[key]].astype(str).map(normalize_full_name_value)

    # Fallback: derive from a file_name-style column (e.g., 'owner.repo__...').
    for key in ("file_name", "filename", "file"):
        if key in cmap:
            def from_file(x: str) -> str:
                base = Path(str(x)).name
                if "__" in base:
                    owner_repo = base.split("__", 1)[0]  # e.g., 'owner.repo'
                    return normalize_full_name_value(owner_repo)
                return ""
            return df[cmap[key]].astype(str).map(from_file)

    raise ValueError("Could not locate a repo identifier column (full_name/repo/… or file_name).")

def repo_variants(owner_slash_repo: str) -> set[str]:
    """
    Return both slash and dot variants for matching:
    - 'owner/repo'
    - 'owner.repo'
    """
    s = owner_slash_repo.strip().lower()
    if not s:
        return set()
    return {s, s.replace("/", ".")}

def infer_full_name_from_config_filename(p: Path) -> str:
    """
    Infer owner/repo from filenames like 'owner.repo__gradle++settings.gradle.kts'.
    Returns '' if not parseable.
    """
    stem = p.name
    if "__" in stem:
        owner_repo = stem.split("__", 1)[0]  # 'owner.repo'
        # normalize to owner/repo
        norm = normalize_full_name_value(owner_repo)
        return norm
    return ""

def scan_text_for_signals(text: str):
    """
    Return: (has_signals [0/1], labels_found, example_snippets)
    """
    labels, snippets = [], []
    for label, pat in SIGNAL_PATTERNS.items():
        m = re.search(pat, text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            labels.append(label)
            s, e = m.span()
            start = max(0, s - 60); end = min(len(text), e + 60)
            snippet = text[start:end].replace("\n", " ").strip()
            snippets.append(f"{label}: …{snippet}…")
    return (1 if labels else 0, labels, snippets)

# ---------- Main ----------
def main():
    # 1) Load CSV and filter Build_pred == 0
    df = load_sample_csv(SAMPLE_FILE)
    build_pred_col = get_build_pred_column(df)
    full_name_series = normalize_full_name_column(df)

    allowed_variants = set()
    for fn, bp in zip(full_name_series, df[build_pred_col]):
        # truthy zero: accept 0, 0.0, "0", " 0 ", False, "false", "False"
        zeroish = False
        if pd.isna(bp):
            zeroish = False
        elif isinstance(bp, (int, float)):
            zeroish = (float(bp) == 0.0)
        else:
            zeroish = str(bp).strip().lower() in {"0", "0.0", "false", "no"}
        if zeroish:
            allowed_variants |= repo_variants(fn)

    print(f"[INFO] Rows in CSV: {len(df)} | Build_pred==0 repos: {len({fn for fn in full_name_series if repo_variants(fn) & allowed_variants})}")

    # 2) Walk config dir and scan only files whose inferred full_name is in 'allowed'
    rows = []
    diag_rows = []
    scanned_files = 0
    matched_files = 0

    for p in CONFIG_DIR.rglob("*"):
        name_l = p.name.lower()
        # Only *.gradle or *.gradle.kts
        if not (name_l.endswith(".gradle") or name_l.endswith(".gradle.kts")):
            continue
        # Exclude exactly build.gradle and build.gradle.kts
        if name_l in ("build.gradle", "build.gradle.kts"):
            continue

        scanned_files += 1

        full_name = infer_full_name_from_config_filename(p)  # returns owner/repo (normalized)
        if not full_name:
            diag_rows.append({"file_path": str(p), "reason": "could_not_infer_repo_from_filename"})
            continue

        # Match against allowed (slash or dot)
        if not (repo_variants(full_name) & allowed_variants):
            diag_rows.append({"file_path": str(p), "reason": f"repo_not_in_Build_pred==0: {full_name}"})
            continue

        matched_files += 1

        try:
            text = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            text = ""

        has_signals, labels, snippets = scan_text_for_signals(text)
        rows.append({
            "full_name": full_name,
            "file_name": p.name,
            "file_path": str(p),
            "has_instru_build_signals": has_signals,
            "signals_found": "; ".join(labels),
            "examples": " | ".join(snippets[:3]),
        })

    print(f"[INFO] Non-build gradle files scanned: {scanned_files} | matched to Build_pred==0 repos: {matched_files}")

    # 3) Save outputs
    cols = ["full_name", "file_name", "file_path", "has_instru_build_signals", "signals_found", "examples"]
    out_df = pd.DataFrame(rows, columns=cols)
    if not out_df.empty:
        out_df = out_df.sort_values(["has_instru_build_signals", "full_name", "file_name"], ascending=[False, True, True])
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUT_CSV, index=False, encoding="utf-8")

    # diagnostics
    diag_df = pd.DataFrame(diag_rows, columns=["file_path", "reason"])
    diag_df.to_csv(OUT_DIAG, index=False, encoding="utf-8")

    print(f"[INFO] Saved results: {OUT_CSV}  (rows={len(out_df)})")
    print(f"[INFO] Saved diagnostics: {OUT_DIAG}  (rows={len(diag_df)})")

if __name__ == "__main__":
    main()


[INFO] Rows in CSV: 383 | Build_pred==0 repos: 171
[INFO] Non-build gradle files scanned: 29449 | matched to Build_pred==0 repos: 1187
[INFO] Saved results: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\instru_signals_in_non_build_gradle_files_Build_pred0.csv  (rows=1187)
[INFO] Saved diagnostics: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\_diagnostics_unmatched_nonbuild_gradle_files.csv  (rows=28262)


Search raw text for instru testing signal in those are not detected by the original search


In [2]:
# -*- coding: utf-8 -*-
"""
Scan CI YAML files (and support scripts) for Android instrumentation-test signals
ONLY for repos whose YAML_pred == 0 (from the input CSV).

Inputs:
- CSV:
    C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\stratified_sample_moe_using RandomSample65.csv
- Configs (archived config files per repo):
    C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files

Output (saved to the Stratified Sample folder):
- 3.2.1_Instru_T_Signal_CI_YAML.csv
"""

from pathlib import Path
import re
import pandas as pd
from typing import List, Tuple

# ------------ Paths ------------
SAMPLE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample")
SAMPLE_FILE = SAMPLE_DIR / "stratified_sample_moe_using RandomSample65.csv"
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")
OUT_CSV = SAMPLE_DIR / "Instru_T_Signal_CI_YAML_Pred0.csv"

# ------------ YAML + Script signal patterns (raw-text; comments included) ------------
YAML_PATTERNS = {
    # GitHub Actions emulator runner
    "uses reactivecircus/android-emulator-runner": r"reactivecircus/android-emulator-runner@",
    # Gradle connected tests via CI steps
    "runs ./gradlew connectedAndroidTest": r"(?m)^\s*[- ]?\s*(?:\.\/|bash\s+\.\/)?gradlew\b.*\bconnected(?:Debug|Release)?AndroidTest\b",
    "runs ./gradlew connectedCheck": r"(?m)^\s*[- ]?\s*(?:\.\/|bash\s+\.\/)?gradlew\b.*\bconnectedCheck\b",
    "assemble*AndroidTest task": r"\bassemble(?:\w+)?AndroidTest\b",
    # Emulator boot by hand
    "sdkmanager system-images": r"\bsdkmanager\b.*\bsystem-images\b",
    "avdmanager create avd": r"\bavdmanager\b.*\bcreate\b.*\bavd\b",
    "emulator -avd": r"\bemulator\s+-avd\b",
    "adb wait-for-device": r"\badb\s+wait-?for-?device\b",
    "wait for boot_completed": r"\bsys\.boot_completed\b",
    # Device farms
    "Firebase Test Lab (gcloud)": r"\bgcloud\s+firebase\s+test\s+android\s+run\b",
    "BrowserStack/AppCenter/SauceLabs": r"(?:browserstack|bstack|app\s*center|sauce\s*labs|saucectl)",
    # Flutter integration tests in CI
    "flutter integration_test/drive": r"flutter\s+(?:test\s+integration_test|drive)\b",
}

SCRIPT_PATTERNS = {
    # Gradle connected tasks in scripts
    "gradlew connectedAndroidTest": r"(?:^|\b)(?:\.\/)?gradlew\b.*\bconnected(?:Debug|Release)?AndroidTest\b",
    "gradlew connectedCheck": r"(?:^|\b)(?:\.\/)?gradlew\b.*\bconnectedCheck\b",
    "assemble*AndroidTest task": r"\bassemble(?:\w+)?AndroidTest\b",
    # Emulator setup in scripts
    "sdkmanager system-images": r"\bsdkmanager\b.*\bsystem-images\b",
    "avdmanager create avd": r"\bavdmanager\b.*\bcreate\b.*\bavd\b",
    "emulator -avd": r"\bemulator\s+-avd\b",
    "adb wait-for-device": r"\badb\s+wait-?for-?device\b",
    "boot_completed": r"\bsys\.boot_completed\b",
    # Device farms
    "gcloud firebase test android run": r"\bgcloud\s+firebase\s+test\s+android\s+run\b",
    "browserstack/appcenter/sauce": r"(?:browserstack|bstack|app\s*center|sauce\s*labs|saucectl)",
    # Flutter integration tests
    "flutter integration_test/drive": r"flutter\s+(?:test\s+integration_test|drive)\b",
}

SCRIPT_EXTENSIONS = {".sh", ".bash", ".cmd", ".bat", ".ps1"}

# ------------ Helpers ------------
def normalize_owner_repo(value: str) -> str:
    """Normalize to 'owner/repo' (lowercase). Accepts 'owner/repo', 'owner.repo', or GitHub URLs."""
    if not value:
        return ""
    s = str(value).strip().replace("\\", "/").lower()
    if "github.com" in s:
        parts = [p for p in s.split("/") if p]
        if len(parts) >= 2:
            owner, repo = parts[-2], parts[-1].removesuffix(".git")
            return f"{owner}/{repo}"
    if "/" in s:
        parts = [p for p in s.split("/") if p]
        if len(parts) >= 2:
            return f"{parts[-2]}/{parts[-1]}"
    if "." in s and "__" not in s:
        parts = [p for p in s.split(".") if p]
        if len(parts) >= 2:
            return f"{parts[-2]}/{parts[-1]}"
    return s

def variants(owner_slash_repo: str) -> set[str]:
    """Return both slash and dot variants for matching."""
    s = owner_slash_repo.strip().lower()
    return {s, s.replace("/", ".")} if s else set()

def infer_owner_repo_from_prefixed_filename(p: Path) -> str:
    """
    Files in the archive are named like 'owner.repo__github_actions++build.yml'.
    Extract 'owner/repo' from the prefix.
    """
    stem = p.name
    if "__" in stem:
        owner_repo = stem.split("__", 1)[0]  # 'owner.repo'
        return normalize_owner_repo(owner_repo)
    return ""

def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df

def find_column_case_insensitive(df: pd.DataFrame, target: str) -> str:
    for c in df.columns:
        if c.lower() == target.lower():
            return c
    raise ValueError(f"Required column '{target}' not found in CSV.")

def zeroish(v) -> bool:
    if pd.isna(v):
        return False
    if isinstance(v, (int, float)):
        return float(v) == 0.0
    return str(v).strip().lower() in {"0", "0.0", "false", "no"}

def scan_text(text: str, patterns: dict) -> List[str]:
    """Return matching pattern labels (raw text, comments included)."""
    if not text:
        return []
    hits = []
    for label, pat in patterns.items():
        if re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL):
            hits.append(label)
    return hits

# ------------ Main ------------
def main():
    df = load_csv(SAMPLE_FILE)
    yaml_col = find_column_case_insensitive(df, "YAML_pred")

    # Repo identifier column: prefer 'full_name', fall back to others (including file_name)
    repo_col = None
    for cand in ("full_name", "repo", "repository", "owner_repo", "name", "file_name", "filename", "file"):
        if cand in df.columns:
            repo_col = cand
            break
    if repo_col is None:
        raise ValueError("CSV must contain a repo column (e.g., 'full_name' or 'repo' or 'file_name').")

    # Build allowed repo set (YAML_pred == 0)
    allowed = set()
    for _, row in df.iterrows():
        if zeroish(row[yaml_col]):
            val = str(row[repo_col])
            # If coming from file_name form like 'owner.repo__...', strip after '__'
            if repo_col.lower() in {"file_name", "filename", "file"} and "__" in val:
                val = val.split("__", 1)[0]
            allowed |= variants(normalize_owner_repo(val))

    rows = []

    # Index scripts per repo (so we can scan support scripts alongside YAMLs)
    scripts_by_repo = {}
    for p in CONFIG_DIR.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() in SCRIPT_EXTENSIONS:
            repo = infer_owner_repo_from_prefixed_filename(p)
            if repo:
                scripts_by_repo.setdefault(repo, []).append(p)

    # Scan YAML files for allowed repos
    for p in CONFIG_DIR.rglob("*"):
        if not p.is_file():
            continue
        name_l = p.name.lower()
        if not (name_l.endswith(".yml") or name_l.endswith(".yaml")):
            continue

        repo = infer_owner_repo_from_prefixed_filename(p)
        if not repo:
            continue
        # Check membership against both slash and dot variants
        if variants(repo) & allowed:
            try:
                ytxt = p.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                ytxt = ""

            yaml_hits = scan_text(ytxt, YAML_PATTERNS)

            # Scan all support scripts for the same repo
            script_hits_detail: List[Tuple[str, List[str]]] = []
            for sp in scripts_by_repo.get(repo, []):
                try:
                    stxt = sp.read_text(encoding="utf-8", errors="ignore")
                except Exception:
                    stxt = ""
                shits = scan_text(stxt, SCRIPT_PATTERNS)
                if shits:
                    script_hits_detail.append((sp.name, shits))

            # Decide YAML_Check
            yaml_check = bool(yaml_hits or script_hits_detail)

            # Build decision reason
            reasons = []
            if yaml_hits:
                reasons.append(f"YAML[{p.name}]: " + ", ".join(yaml_hits))
            if script_hits_detail:
                for sname, shits in script_hits_detail:
                    reasons.append(f"SCRIPT[{sname}]: " + ", ".join(shits))
            if not reasons:
                reasons.append("No emulator/connected-test/device-farm/flutter-integration signals found in YAML or scripts")

            rows.append({
                "full_name": repo,                # normalized owner/repo
                "file_name": p.name,              # the YAML file
                "YAML_Check": bool(yaml_check),
                "decision_reason": " | ".join(reasons)
            })

    # If some allowed repos had no YAML files at all, record a row so you know they were considered
    yaml_repos_seen = {r["full_name"] for r in rows}
    for repo_variant in allowed:
        # normalize variant back to slash form for comparison
        repo_slash = repo_variant.replace(".", "/")
        if repo_slash not in yaml_repos_seen:
            rows.append({
                "full_name": repo_slash,
                "file_name": "",
                "YAML_Check": False,
                "decision_reason": "No YAML file found under All_Config_Files for this repo"
            })

    out = pd.DataFrame(rows, columns=["full_name", "file_name", "YAML_Check", "decision_reason"])
    # Sort: positives first, then repo, then file name
    if not out.empty:
        out = out.sort_values(["YAML_Check", "full_name", "file_name"], ascending=[False, True, True])

    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT_CSV, index=False, encoding="utf-8")
    print(f"[INFO] Saved: {OUT_CSV} (rows={len(out)})")

if __name__ == "__main__":
    main()


[INFO] Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Instru_T_Signal_CI_YAML_Pred0.csv (rows=756)
